# jaxoom T4 cross-device validation

This notebook validates the frozen JAX 0.11 GPU memory calibration on an independent NVIDIA Tesla T4.

Before running:

1. Runtime -> Change runtime type -> T4 GPU
2. Runtime -> Run all

The final cell downloads `jaxoom_t4_validation_results.zip`. No credentials or Google Drive are required.


In [ ]:
# Hardware preflight. This cell intentionally does not import JAX.
from datetime import datetime, timezone
import json
import platform
import subprocess

query = [
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader,nounits",
]
probe = subprocess.run(query, capture_output=True, text=True)
if probe.returncode != 0:
    raise RuntimeError("nvidia-smi failed. Select a Colab T4 GPU runtime before running all cells.\n" + probe.stderr[-2000:])
print(probe.stdout.strip())
if not probe.stdout.strip():
    raise RuntimeError("No NVIDIA GPU was reported by nvidia-smi.")
PREINSTALL_GPU_QUERY = probe.stdout.strip()
full_probe = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
PREINSTALL_NVIDIA_SMI = full_probe.stdout


In [ ]:
# Install the exact CUDA 12 JAX 0.11.0 package set.
# The JAX 0.11.0 PyPI metadata provides the cuda12 extra and pins compatible
# jaxlib and jax-cuda12-plugin releases. JAX is not imported before this cell.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--upgrade",
    "jax[cuda12]==0.11.0",
])
print("Installed pinned JAX CUDA 12 package specification: jax[cuda12]==0.11.0")


In [ ]:
# Validate the installed JAX runtime and require an actual Tesla T4.
import importlib.metadata
import os
import re
import sys

import jax
import jaxlib

if jax.__version__ != "0.11.0":
    raise RuntimeError(f"Expected JAX 0.11.0, found {jax.__version__}")
if jax.default_backend() != "gpu":
    raise RuntimeError(f"Expected GPU backend, found {jax.default_backend()}")
devices = jax.devices()
if not devices:
    raise RuntimeError("JAX reported no devices.")
if any("t4" not in str(getattr(device, "device_kind", device)).lower() and "t4" not in str(device).lower() for device in devices):
    raise RuntimeError(
        "This notebook requires a Google Colab Tesla T4 runtime. "
        f"Current devices: {devices}"
    )
if len(devices) != 1:
    raise RuntimeError(f"Expected one Colab GPU device, found {len(devices)}: {devices}")

def nvidia_query(query):
    result = subprocess.run(
        ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()

GPU_NAME = nvidia_query("name")
if "t4" not in GPU_NAME.lower():
    raise RuntimeError(f"This notebook requires a Tesla T4. nvidia-smi reported: {GPU_NAME}")

ENVIRONMENT = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(),
    "python_version": platform.python_version(),
    "jax_version": jax.__version__,
    "jaxlib_version": jaxlib.__version__,
    "backend": jax.default_backend(),
    "devices": [str(device) for device in devices],
    "device_kind": [getattr(device, "device_kind", None) for device in devices],
    "gpu_name": GPU_NAME,
    "gpu_memory_mib": nvidia_query("memory.total"),
    "driver_version": nvidia_query("driver_version"),
    "cuda_driver_version": (re.search(r"CUDA Version:\s*([^|\n]+)", PREINSTALL_NVIDIA_SMI) or [None, None])[1],
    "colab_detected": os.path.exists("/content") and "COLAB_GPU" in os.environ,
    "allocator_environment": {
        key: os.environ.get(key)
        for key in (
            "XLA_PYTHON_CLIENT_PREALLOCATE",
            "XLA_PYTHON_CLIENT_MEM_FRACTION",
            "XLA_CLIENT_MEM_FRACTION",
            "XLA_PYTHON_CLIENT_ALLOCATOR",
            "TF_GPU_ALLOCATOR",
        )
    },
}
print(json.dumps(ENVIRONMENT, indent=2, sort_keys=True))


In [ ]:
# Clone the exact public jaxoom revision used for this experiment.
from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path("/content/jaxoom")
PINNED_COMMIT = "c5925ff7404953c3640756d9b15bea898e60477d"
REPOSITORY = "https://github.com/Slavov88/jaxoom.git"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.check_call(["git", "clone", "--quiet", REPOSITORY, str(REPO_DIR)])
subprocess.check_call(["git", "checkout", "--quiet", PINNED_COMMIT], cwd=REPO_DIR)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if actual_commit != PINNED_COMMIT:
    raise RuntimeError(f"Pinned revision mismatch: {actual_commit}")
if not (REPO_DIR / "experiments" / "device_transfer_validation.py").exists():
    raise RuntimeError("Pinned revision does not contain the device-transfer harness.")
print(f"Pinned jaxoom revision: {actual_commit}")


In [ ]:
# Install jaxoom without allowing pip to change the pinned JAX stack.
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO_DIR)])


In [ ]:
# Validate the public API and load the committed RTX 3050 baseline artifact.
import json
import sys
from pathlib import Path

import jax.numpy as jnp
import jaxoom

sys.path.insert(0, str(REPO_DIR / "experiments"))
BASELINE_PATH = REPO_DIR / "experiments" / "jax_0_11_gpu_calibration_2026-09-08_v2.json"
if not BASELINE_PATH.exists():
    raise RuntimeError(f"Missing committed baseline artifact: {BASELINE_PATH}")
baseline_payload = json.loads(BASELINE_PATH.read_text(encoding="utf-8"))
baseline_rows = baseline_payload.get("rows", [])
if len(baseline_rows) != 46:
    raise RuntimeError(f"Expected 46 baseline cases, found {len(baseline_rows)}")
if len({row["name"] for row in baseline_rows}) != 46:
    raise RuntimeError("Baseline contains duplicate case IDs")
if len({row["family"] for row in baseline_rows}) != 9:
    raise RuntimeError("Baseline does not contain the expected 9 workload families")
if sum(row["dtype"] == "float32" for row in baseline_rows) != 23 or sum(row["dtype"] == "float16" for row in baseline_rows) != 23:
    raise RuntimeError("Baseline dtype counts are not 23 float32 and 23 float16")

# Use the public calls with an explicit JAX abstract input.
import jax
abstract = jax.ShapeDtypeStruct((8, 8), jnp.float32)
static = jaxoom.estimate(lambda value: jnp.sin(value) + value, abstract)
compiler = jaxoom.compile_analyze(lambda value: jnp.sin(value) + value, abstract)
comparison = jaxoom.compare_memory(static, compiler)
interval = jaxoom.calibrate(static)
assessment = jaxoom.assess(static, "16 GiB")
if interval.applicability != "EXACT_TESTED":
    raise RuntimeError(f"Expected exact frozen calibration, found {interval.applicability}")
print({
    "estimate": static.estimated_peak_bytes,
    "compiler": compiler.compiler_accounted_bytes,
    "comparison": comparison.compiler_accounted_bytes,
    "calibration": interval.dataset_version,
    "applicability": interval.applicability,
    "assessment_calibrated": assessment.calibrated,
})


In [ ]:
# Run three representative smoke cases before the full matrix.
import json
import sys

from accelerator_calibration import cases, environment, run_case

case_list = cases()
by_family = {}
for case in case_list:
    if case.dtype == "float32" and case.family not in by_family:
        by_family[case.family] = case
smoke_cases = [by_family[family] for family in ("elementwise", "mlp", "attention")]
smoke_environment = environment()
if "t4" not in str(smoke_environment.get("device_kind", "")).lower() and "t4" not in str(smoke_environment.get("device", "")).lower():
    raise RuntimeError(f"Device changed before smoke validation: {smoke_environment}")
smoke_rows = []
for case in smoke_cases:
    row = run_case(case, smoke_environment)
    smoke_rows.append({key: row.get(key) for key in ("name", "family", "dtype", "status", "static_peak_bytes", "compiler_accounted_bytes", "compiler_temp_bytes", "error_message")})
    if row.get("status") != "ok":
        raise RuntimeError(f"Smoke case failed: {row}")
print(json.dumps(smoke_rows, indent=2, sort_keys=True))


In [ ]:
# Run the full 46-case paired device-transfer matrix.
import os
import subprocess
from pathlib import Path

FULL_RUN = True
RESULT_DIR = Path("/content/jaxoom_t4_results")
if RESULT_DIR.exists():
    import shutil
    shutil.rmtree(RESULT_DIR)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PREFIX = RESULT_DIR / "t4_device_transfer_validation"
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO_DIR / 'src'}:{REPO_DIR / 'experiments'}"
command = [
    sys.executable,
    str(REPO_DIR / "experiments" / "device_transfer_validation.py"),
    "--reference",
    str(BASELINE_PATH),
    "--output-prefix",
    str(OUTPUT_PREFIX),
]
if not FULL_RUN:
    command.extend(["--limit", "3", "--allow-same-device-smoke"])
completed = subprocess.run(command, cwd=REPO_DIR, env=env, text=True, capture_output=True)
if completed.returncode != 0:
    print(completed.stdout)
    print(completed.stderr[-4000:])
    raise RuntimeError(f"Device-transfer harness failed with exit code {completed.returncode}")
print(completed.stdout.strip())


In [ ]:
# Validate raw result integrity and enrich the report with notebook metadata.
import json
from pathlib import Path

raw_path = RESULT_DIR / "t4_device_transfer_validation.json"
summary_path = RESULT_DIR / "t4_device_transfer_validation_summary.json"
report_path = RESULT_DIR / "t4_device_transfer_validation_report.md"
payload = json.loads(raw_path.read_text(encoding="utf-8"))
summary = json.loads(summary_path.read_text(encoding="utf-8"))
raw_cases = payload.get("cases", [])
baseline_by_name = {row["name"]: row for row in baseline_rows}
if len(raw_cases) != 46:
    raise RuntimeError(f"Expected 46 attempted T4 cases, found {len(raw_cases)}")
if len({row["name"] for row in raw_cases}) != 46:
    raise RuntimeError("T4 result contains duplicate case IDs")
if set(row["name"] for row in raw_cases) != set(baseline_by_name):
    raise RuntimeError("T4 case IDs do not match the RTX 3050 baseline")
if payload.get("independent_device") is not True:
    raise RuntimeError("Harness did not classify the T4 as an independent device")
if payload.get("current_environment", {}).get("jax_version") != "0.11.0":
    raise RuntimeError("T4 result does not use JAX 0.11.0")
if payload.get("current_environment", {}).get("backend") != "gpu":
    raise RuntimeError("T4 result does not use the GPU backend")

successful = [row for row in raw_cases if row.get("current_compiler_bytes") is not None]
static_mismatches = [row for row in successful if row["static_bytes_current"] != row["static_bytes_reference"]]
if static_mismatches:
    print("Static mismatches detected:", [row["name"] for row in static_mismatches])

failures = [row for row in raw_cases if row.get("current_compiler_bytes") is None]
if failures:
    print("Incomplete cases:", json.dumps(failures, indent=2, sort_keys=True))

ENVIRONMENT["pinned_repository_commit"] = PINNED_COMMIT
ENVIRONMENT["baseline_artifact"] = str(BASELINE_PATH)
(RESULT_DIR / "t4_environment.json").write_text(json.dumps(ENVIRONMENT, indent=2, sort_keys=True) + "\n", encoding="utf-8")

transfer_label = "INCOMPLETE" if failures else ("GOOD_TRANSFER" if summary["upper_coverage"] >= 0.90 else "MIXED_TRANSFER" if summary["upper_coverage"] >= 0.80 else "POOR_TRANSFER")
report = f'''# jaxoom T4 cross-device validation

**Status: {'COMPUTATIONALLY VERIFIED' if not failures else 'INCOMPLETE'}.**

## Environment

- Source GPU: NVIDIA GeForce RTX 3050 Laptop GPU
- Validation GPU: {ENVIRONMENT['gpu_name']}
- VRAM: {ENVIRONMENT['gpu_memory_mib']} MiB
- Driver: {ENVIRONMENT['driver_version']}
- Python: {ENVIRONMENT['python_version']}
- JAX: {ENVIRONMENT['jax_version']}
- jaxlib: {ENVIRONMENT['jaxlib_version']}
- Backend: {ENVIRONMENT['backend']}
- Pinned repository commit: `{PINNED_COMMIT}`

## Matched dataset

- Attempted cases: {len(raw_cases)}
- Successful compiler cases: {len(successful)}
- Failed cases: {len(failures)}
- Families: {len({row['family'] for row in baseline_rows})}
- float32: {sum(row['dtype'] == 'float32' for row in baseline_rows)}
- float16: {sum(row['dtype'] == 'float16' for row in baseline_rows)}
- Static matches: {len(successful) - len(static_mismatches)}/{len(successful)}

## Frozen calibration transfer

- Dataset: `{payload['frozen_calibration']['dataset_version']}`
- Lower ratio: {payload['frozen_calibration']['lower_ratio']:.6f}
- Central ratio: {payload['frozen_calibration']['central_ratio']:.6f}
- Upper ratio: {payload['frozen_calibration']['upper_ratio']:.6f}
- Interval coverage: {summary['interval_coverage_count']}/{summary['sample_count']} = {summary['interval_coverage']:.1%}
- Upper coverage: {summary['upper_coverage_count']}/{summary['sample_count']} = {summary['upper_coverage']:.1%}
- Upper miss rate: {summary['upper_miss_rate']:.1%}
- Median width/static: {summary['median_width_over_static']:.4f}
- Experiment label: `{transfer_label}`

## Compiler drift

- Median absolute relative drift: {summary['median_absolute_relative_drift']}
- P90 absolute relative drift: {summary['p90_absolute_relative_drift']}
- Maximum absolute relative drift: {summary['maximum_absolute_relative_drift']}
- Temporary ratio median/P90/maximum: {summary['temporary_ratio_median']}, {summary['temporary_ratio_p90']}, {summary['temporary_ratio_maximum']}
- Temporary zero to nonzero cases: {summary['temporary_zero_to_nonzero']}
- Temporary nonzero to zero cases: {summary['temporary_nonzero_to_zero']}
- Nonzero alias deltas: {summary['alias_delta_nonzero']}

## Family results

```json
{json.dumps(summary['by_family'], indent=2, sort_keys=True)}
```

## Dtype results

```json
{json.dumps(summary['by_dtype'], indent=2, sort_keys=True)}
```

## Limitations

This result evaluates compiler accounting on one Colab Tesla T4 against one
RTX 3050 Laptop GPU using JAX 0.11.0. It does not modify calibration constants,
claim universal NVIDIA support, or establish exact runtime peak memory.
'''
report_path.write_text(report, encoding="utf-8")
print(report)


In [ ]:
# Optional runtime sanity smoke using the existing experimental harness.
import json
import subprocess

runtime_specs = [
    ("attention", {"sequence": 512, "heads": 8, "head_dim": 64}),
    ("mlp", {"batch": 256, "width": 1024, "depth": 2}),
    ("training", {"batch": 128, "width": 512}),
]
runtime_rows = []
for family, config in runtime_specs:
    command = [
        sys.executable,
        str(REPO_DIR / "experiments" / "execution_memory_validation.py"),
        "--trial", family,
        "--config", json.dumps(config, sort_keys=True),
        "--dtype", "float32",
        "--repetitions", "2",
        "--poll-interval", "0.002",
    ]
    completed = subprocess.run(command, cwd=REPO_DIR, env=env, text=True, capture_output=True, timeout=300)
    if completed.returncode == 0:
        try:
            runtime_rows.append(json.loads(completed.stdout))
        except json.JSONDecodeError:
            runtime_rows.append({"family": family, "configuration": config, "status": "FAIL", "failure_message": "invalid harness JSON"})
    else:
        runtime_rows.append({"family": family, "configuration": config, "status": "FAIL", "failure_message": completed.stderr[-2000:]})
(RESULT_DIR / "t4_runtime_smoke.json").write_text(json.dumps(runtime_rows, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps([{key: row.get(key) for key in ("family", "status", "structural_peak_bytes", "calibrated_upper_bytes", "compiler_accounted_bytes", "execution_peak_bytes")} for row in runtime_rows], indent=2))


In [ ]:
# Package only compact result artifacts and validate the final archive.
import shutil
import zipfile

zip_path = Path("/content/jaxoom_t4_validation_results.zip")
if zip_path.exists():
    zip_path.unlink()
artifact_names = [
    "t4_device_transfer_validation.json",
    "t4_device_transfer_validation_summary.json",
    "t4_device_transfer_validation_report.md",
    "t4_environment.json",
    "t4_runtime_smoke.json",
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in artifact_names:
        path = RESULT_DIR / name
        if not path.exists():
            raise RuntimeError(f"Missing artifact: {path}")
        archive.write(path, arcname=name)
with zipfile.ZipFile(zip_path) as archive:
    if set(archive.namelist()) != set(artifact_names):
        raise RuntimeError("ZIP contents do not match the compact artifact list")
print(f"Created {zip_path} ({zip_path.stat().st_size} bytes)")


In [ ]:
# Download the result bundle.
from google.colab import files

print("Download path:", zip_path)
files.download(str(zip_path))
